**A Comparative Study of BERT, RoBERTa, and DistilBERT against a Classical Machine Learning Models for Mental Health Mental Health Discourse Classification**

1. Importing necessary libraries

In [1]:
import os, re, json, time, warnings, logging

import nltk
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,confusion_matrix, f1_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,get_linear_schedule_with_warmup)

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

2.  Config - 

In [2]:
DATA_PATH  = "/kaggle/input/datasets/ushapoudellamgade/mental-health-data/data.csv"
OUTPUT_DIR = "/kaggle/working/outputs"
SEED = 42

# Transformer hyper-parameters 
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 10         
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.10
GRAD_CLIP = 1.0

# ── Split ratios: 70 / 15 / 15
TEST_SIZE = 0.15
VAL_SIZE  = 0.15

# TF-IDF settings 
TFIDF_MAX_FEATURES = 50_000
TFIDF_NGRAM_RANGE  = (1, 2)

EXPECTED_CLASSES = ["Normal", "Depression", "Suicidal","Anxiety", "Bipolar", "Stress", "Personality Disorder",]

MODEL_ORDER = ["Naive Bayes", "Logistic Regression", "Linear SVC","DistilBERT", "BERT", "RoBERTa"]

MODEL_PARAMS = {"Naive Bayes": "—", "Logistic Regression": "—", "Linear SVC": "—","DistilBERT": "66M", "BERT": "110M", "RoBERTa": "125M"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

3. NLTK Resources

In [3]:
for _pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(_pkg, quiet=True)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

4. Data Loading and Exploratory Data Analysis (EDA)

In [4]:
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=["statement", "status"])
    df.dropna(inplace=True)
    df.rename(columns={"statement": "text", "status": "label"}, inplace=True)
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.len() > 2]
    return df.reset_index(drop=True)


def print_class_distribution(df: pd.DataFrame):
    counts = df["label"].value_counts()
    total  = len(df)
    print("\n" + "=" * 55)
    print("  TABLE I — Class Distribution")
    print("=" * 55)
    print(f"  {'Category':<25} {'Samples':>8}  {'%':>6}")
    print("-" * 55)
    for label, count in counts.items():
        print(f"  {label:<25} {count:>8,}  {100 * count / total:>5.1f}")
    print("-" * 55)
    print(f"  {'Total':<25} {total:>8,}  {'100.0':>6}")
    print("=" * 55 + "\n")


def plot_class_distribution(df: pd.DataFrame):
    counts = df["label"].value_counts().reindex(EXPECTED_CLASSES, fill_value=0)
    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(counts.index, counts.values,
                  color=sns.color_palette("Set2", len(counts)))
    ax.set_title("Table I — Class Distribution", fontsize=13, fontweight="bold")
    ax.set_xlabel("Mental Health Category")
    ax.set_ylabel("Number of Samples")
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 80, f"{val:,}",
                ha="center", va="bottom", fontsize=8)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/class_distribution.png", dpi=150)
    plt.close()


def plot_text_length(df: pd.DataFrame):
    temp = df.copy()
    temp["length"] = temp["text"].apply(lambda x: len(x.split()))
    fig, ax = plt.subplots(figsize=(11, 4))
    for label in EXPECTED_CLASSES:
        subset = temp[temp["label"] == label]["length"]
        if len(subset):
            ax.hist(subset, bins=50, alpha=0.55, label=label)
    ax.axvline(MAX_LEN, color="red", linestyle="--",
               linewidth=1.5, label=f"MAX_LEN={MAX_LEN}")
    ax.set_title("Token Length Distribution by Class", fontsize=12)
    ax.set_xlabel("Number of Words")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/text_length_distribution.png", dpi=150)
    plt.close()

5. Preprocessing

In [5]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [lemmatizer.lemmatize(t) for t in text.split()
              if t not in stop_words and len(t) > 2]
    return " ".join(tokens)


def preprocess(df: pd.DataFrame):
    print("[PRE] Cleaning text for classical models …")
    df = df.copy()
    df["clean_text"] = df["text"].apply(clean_text)
    le = LabelEncoder()
    df["label_id"] = le.fit_transform(df["label"])
    return df, le

6. Classical Baselines

In [6]:
def run_classical_models(X_tr, y_tr, X_te, y_te, class_names: list) -> dict:
    results = {}
    vectorizer = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES,ngram_range=TFIDF_NGRAM_RANGE,sublinear_tf=True)
    X_tr_tf = vectorizer.fit_transform(list(X_tr))
    X_te_tf = vectorizer.transform(list(X_te))

    models = {"Logistic Regression": LogisticRegression(max_iter=1000, C=1.0,class_weight="balanced",random_state=SEED),
        "Naive Bayes":  MultinomialNB(alpha=0.1),"Linear SVC":   LinearSVC(C=1.0, class_weight="balanced",max_iter=2000, random_state=SEED)}

    for name, clf in models.items():
        t0 = time.time()
        clf.fit(X_tr_tf, y_tr)
        preds   = clf.predict(X_te_tf)
        elapsed = time.time() - t0
        acc = accuracy_score(y_te, preds)
        f1  = f1_score(y_te, preds, average="weighted")
        print(f"[CLS] {name:<22}  Acc={acc:.4f}  F1={f1:.4f}  ({elapsed:.1f}s)")
        results[name] = {"accuracy":    acc,"f1_weighted": f1,"report":classification_report(y_te, preds,target_names=class_names),"cm": confusion_matrix(y_te, preds)}
    return results

7. Pytorch Dataset

In [7]:
class MHDataset(Dataset):
    def __init__(self, texts: list, labels: list, tokenizer):
        self.enc = tokenizer(texts, truncation=True, padding="max_length",max_length=MAX_LEN, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}, self.labels[i]


8. Transformer fine tuning

In [8]:
def _make_loaders(X_tr, y_tr, X_val, y_val, X_te, y_te, tokenizer, device):
    """Build DataLoaders; pin_memory only when CUDA is available (FIX #2)."""
    pin = device.type == "cuda"   # FIX #2: pin_memory=True only for GPU

    tr_ds  = MHDataset(X_tr.tolist(),  y_tr.tolist(),  tokenizer)
    val_ds = MHDataset(X_val.tolist(), y_val.tolist(), tokenizer)
    te_ds  = MHDataset(X_te.tolist(),  y_te.tolist(),  tokenizer)

    tr_dl  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=pin)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE,
                        num_workers=0, pin_memory=pin)
    te_dl  = DataLoader(te_ds,  batch_size=BATCH_SIZE,
                        num_workers=0, pin_memory=pin)
    return tr_dl, val_dl, te_dl


def _train_and_evaluate(model, loss_fn, optimizer, scheduler,
                        tr_dl, val_dl, te_dl, device, model_label):
    """
    Core training loop. Returns result dict with accuracy, f1, cm, history.
    Raises RuntimeError on unrecoverable failure so the caller can retry on CPU.
    """
    best_val_acc = 0.0
    best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    history      = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(EPOCHS):
        # train
        model.train()
        tr_loss = 0.0
        for batch_enc, batch_lbl in tr_dl:
            batch_enc = {k: v.to(device) for k, v in batch_enc.items()}
            batch_lbl = batch_lbl.to(device)
            optimizer.zero_grad()
            out  = model(**batch_enc)
            loss = loss_fn(out.logits, batch_lbl)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            tr_loss += loss.item()

        # validate 
        model.eval()
        val_loss, val_preds, val_true = 0.0, [], []
        with torch.no_grad():
            for batch_enc, batch_lbl in val_dl:
                batch_enc = {k: v.to(device) for k, v in batch_enc.items()}
                batch_lbl = batch_lbl.to(device)
                out = model(**batch_enc)
                val_loss += loss_fn(out.logits, batch_lbl).item()
                val_preds.extend(out.logits.argmax(-1).cpu().tolist())
                val_true.extend(batch_lbl.cpu().tolist())

        avg_tr  = tr_loss  / len(tr_dl)
        avg_val = val_loss / len(val_dl)
        val_acc = accuracy_score(val_true, val_preds)
        history["train_loss"].append(avg_tr)
        history["val_loss"].append(avg_val)
        history["val_acc"].append(val_acc)
        print(f"  Epoch {epoch+1}/{EPOCHS}  "
              f"tr_loss={avg_tr:.4f}  val_loss={avg_val:.4f}  "
              f"val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}

    # test
    model.load_state_dict(best_state)
    model.to(device).eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch_enc, batch_lbl in te_dl:
            batch_enc = {k: v.to(device) for k, v in batch_enc.items()}
            preds.extend(model(**batch_enc).logits.argmax(-1).cpu().tolist())
            trues.extend(batch_lbl.tolist())

    acc = accuracy_score(trues, preds)
    f1  = f1_score(trues, preds, average="weighted")
    print(f"[TRF] {model_label}  TEST → Acc={acc:.4f}  F1={f1:.4f}")

    return {"accuracy":acc,"f1_weighted": f1,"preds":preds,"trues":trues,"history":history} 

def run_transformer(model_name, X_tr, y_tr, X_val, y_val, X_te, y_te,
                    num_labels, class_names) -> dict | None:
    """
    Downloads tokenizer + model, fine-tunes, evaluates on test split.
    FIX #3: If CUDA OOM occurs, automatically retries on CPU.
    Returns result dict (always includes accuracy + f1_weighted) or None.
    """

    # tokenizer 
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception as e:
        print(f"[TRF] ✗ Tokenizer download failed for {model_name}: {e}")
        return None

    # helper: build model + optimizer for a given device 
    def _setup(device):
        y_tr_np = y_tr.values if hasattr(y_tr, "values") else np.array(y_tr)
        weights = compute_class_weight("balanced",classes=np.unique(y_tr_np), y=y_tr_np)
        class_w = torch.tensor(weights, dtype=torch.float).to(device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_w)

        mdl = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels,
            ignore_mismatched_sizes=True).to(device)

        opt   = AdamW(mdl.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        tr_dl, val_dl, te_dl = _make_loaders(
            X_tr, y_tr, X_val, y_val, X_te, y_te, tokenizer, device)
        total_steps = len(tr_dl) * EPOCHS
        sched = get_linear_schedule_with_warmup(
            opt,
            num_warmup_steps=int(WARMUP_FRACTION * total_steps),
            num_training_steps=total_steps)
        return mdl, loss_fn, opt, sched, tr_dl, val_dl, te_dl

    # attempt GPU first, fall back to CPU on OOM 
    for attempt, use_cuda in enumerate([True, False]):
        if use_cuda and not torch.cuda.is_available():
            continue

        device = torch.device("cuda" if use_cuda else "cpu")
        print(f"[TRF] {model_name}  |  device={device}"
              + ("  (CPU fallback)" if attempt == 1 else ""))
        try:
            mdl, loss_fn, opt, sched, tr_dl, val_dl, te_dl = _setup(device)
            raw = _train_and_evaluate(mdl, loss_fn, opt, sched,tr_dl, val_dl, te_dl, device, model_name)
            break   # success — exit retry loop

        except torch.cuda.OutOfMemoryError:
            # FIX #3: OOM on GPU → free memory and retry on CPU
            print(f"[TRF] CUDA OOM on {model_name} — retrying on CPU …")
            torch.cuda.empty_cache()
            continue

        except Exception as e:
            print(f"[TRF] ✗ {model_name} failed: {e}")
            return None
    else:
        # Both GPU and CPU failed (or only CPU attempted and failed)
        print(f"[TRF] ✗ {model_name} could not complete on any device.")
        return None

    # Build classification report + confusion matrix from stored preds
    report = classification_report(raw["trues"], raw["preds"],target_names=class_names)
    cm     = confusion_matrix(raw["trues"], raw["preds"])
    return {"accuracy":raw["accuracy"],"f1_weighted": raw["f1_weighted"],  
        "report": report,"cm": cm,
        "history":raw["history"]}

9. Plotting helper

In [9]:
def _safe_filename(name: str) -> str:
    return name.replace(" ", "_").replace("/", "_")


def plot_confusion_matrix(cm, class_names, title, filename):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{filename}", dpi=150)
    plt.close()


def plot_all_confusion_matrices(all_results: dict, class_names: list):
    """
    Single figure with one confusion-matrix heatmap per model that completed.
    Layout: 2 rows × 3 cols (classical top row, transformers bottom row).
    Rows are labelled; each panel shows model name + Acc/F1 in the title.
    Saved as  all_confusion_matrices.png
    """
    present = [n for n in MODEL_ORDER if n in all_results]
    n= len(present)
    if n == 0:
        return

    # Always use 2 rows × 3 cols — empty cells are hidden if < 6 models ran
    ncols, nrows = 3, 2
    fig, axes = plt.subplots(nrows, ncols,figsize=(ncols * 7, nrows * 6),constrained_layout=True,)
    axes_flat = axes.flatten()

    # Colour palette: one distinct colour per class for the diagonal highlight
    # (seaborn heatmap uses a single cmap; we annotate instead)
    cmaps = {"Naive Bayes":"Purples","Logistic Regression":"Greens",
        "Linear SVC":"Oranges","DistilBERT":"Blues",
        "BERT":"Blues","RoBERTa":"Blues"}

    for idx, name in enumerate(present):
        ax  = axes_flat[idx]
        res = all_results[name]
        cm  = res["cm"]

        # Normalised version (percentages) shown as annotations alongside counts
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

        # Build annotation strings: "N\n(xx.x%)"
        annot = np.empty_like(cm, dtype=object)
        for r in range(cm.shape[0]):
            for c in range(cm.shape[1]):
                annot[r, c] = f"{cm[r, c]}\n({cm_norm[r, c]:.1f}%)"

        cmap = cmaps.get(name, "Blues")
        sns.heatmap(cm_norm,annot=annot,fmt="",cmap=cmap,vmin=0, vmax=100,xticklabels=class_names,yticklabels=class_names,linewidths=0.4,linecolor="white",cbar_kws={"label": "Row %", "shrink": 0.75},ax=ax)
        acc = res["accuracy"]
        f1  = res["f1_weighted"]
        model_type = "Classical" if name in ("Naive Bayes","Logistic Regression","Linear SVC") else "Transformer"
        ax.set_title(f"{name}  [{model_type}]\n"
            f"Acc = {acc:.4f}   F1 (wtd) = {f1:.4f}",
            fontsize=10, fontweight="bold", pad=8,)
        ax.set_xlabel("Predicted Label",fontsize=8)
        ax.set_ylabel("True Label",fontsize=8)
        ax.tick_params(axis="x", labelsize=7, rotation=35)
        ax.tick_params(axis="y", labelsize=7, rotation=0)

    # Hide unused subplots (if fewer than 6 models ran)
    for idx in range(len(present), ncols * nrows):
        axes_flat[idx].set_visible(False)

    # Super-title
    fig.suptitle("Figure — Confusion Matrices for All 6 Models (Test Set)\n"
        "Colour intensity = row-normalised % · Annotation = count (row %)",fontsize=13, fontweight="bold", y=1.02,)

    out_path = f"{OUTPUT_DIR}/all_confusion_matrices.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[PLT] all_confusion_matrices.png saved  ({len(present)}/6 models).")


def plot_training_history(history, model_short):
    epochs_range = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs_range, history["train_loss"],"b-o", label="Train Loss")
    ax1.plot(epochs_range, history["val_loss"],"r-o", label="Val Loss")
    ax1.set_title(f"{model_short} — Loss Curve")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()
    ax2.plot(epochs_range, history["val_acc"], "g-o", label="Val Accuracy")
    ax2.set_title(f"{model_short} — Validation Accuracy")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend()
    plt.tight_layout()
    plt.savefig(
        f"{OUTPUT_DIR}/{_safe_filename(model_short)}_training_history.png",
        dpi=150)
    plt.close()


def plot_model_comparison(all_results: dict):
    names = [n for n in MODEL_ORDER if n in all_results]
    accs  = [all_results[n]["accuracy"]    for n in names]
    f1s   = [all_results[n]["f1_weighted"] for n in names]

    x, width = np.arange(len(names)), 0.35
    fig, ax = plt.subplots(figsize=(13, 6))
    b1 = ax.bar(x - width/2, accs, width, label="Accuracy",      color="#4472C4")
    b2 = ax.bar(x + width/2, f1s,  width, label="F1 (weighted)", color="#ED7D31")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha="right", fontsize=10)
    ax.set_ylim(0, 1.10)
    ax.set_title("Table II — Model Comparison: Accuracy & Weighted F1",fontsize=13, fontweight="bold")
    ax.set_ylabel("Score"); ax.legend()
    for bar in list(b1) + list(b2):
        ax.text(bar.get_x() + bar.get_width()/2,bar.get_height() + 0.005,f"{bar.get_height():.3f}",ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/model_comparison.png", dpi=150)
    plt.close()
    print("[PLT] model_comparison.png saved.")


def print_final_table(all_results: dict):
    """FIX #5: Prints F1 + Accuracy for ALL 6 models (or flags missing ones)."""
    print("\n" + "=" * 68)
    print("  TABLE II — Model Performance on Test Set")
    print("=" * 68)
    print(f"  {'Model':<22}  {'Accuracy':>10}  {'F1 (wtd)':>10}  {'Params':>8}")
    print("-" * 68)
    for name in MODEL_ORDER:
        if name not in all_results:
            print(f"  {name:<22}  {'(not run)':>10}  {'':>10}  "
                  f"{MODEL_PARAMS.get(name,'—'):>8}")
            continue
        res = all_results[name]
        print(f"  {name:<22}  {res['accuracy']:>10.4f}"
              f"{res['f1_weighted']:>10.4f}  {MODEL_PARAMS.get(name,'—'):>8}")
    print("=" * 68 + "\n")



10. Main pipeline

In [10]:
def main():
    print("=" * 70)
    print("Mental Health Discourse Classification — IEEE Paper Pipeline")
    print("6 Models: LR | NB | LinearSVC | DistilBERT | BERT | RoBERTa")
    print("=" * 70)

    # Step 1: Load & EDA 
    df = load_data(DATA_PATH)
    print(f"[DATA] Loaded {len(df):,} rows | {df['label'].nunique()} classes")
    print_class_distribution(df)
    plot_class_distribution(df)
    plot_text_length(df)

    # Step 2: Preprocess 
    df, le = preprocess(df)
    class_names = list(le.classes_)
    num_labels  = len(class_names)
    print(f"[PRE] Classes ({num_labels}): {class_names}")

    # Step 3: ONE stratified split — ALL models use the SAME test set 
    X_tmp, X_te, y_tmp, y_te = train_test_split(
        df[["clean_text", "text"]], df["label_id"],
        test_size=TEST_SIZE, random_state=SEED, stratify=df["label_id"])
    val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
    X_tr, X_val, y_tr, y_val = train_test_split(X_tmp, y_tmp,test_size=val_ratio, random_state=SEED, stratify=y_tmp)

    for frame in [X_tr, X_val, X_te, y_tr, y_val, y_te]:
        frame.reset_index(drop=True, inplace=True)

    total = len(X_tr) + len(X_val) + len(X_te)
    print(f"[SPL] Train={len(X_tr):,}Val={len(X_val):,}  Test={len(X_te):,}"
          f"({100*len(X_tr)/total:.1f}% / {100*len(X_val)/total:.1f}% /"
          f"{100*len(X_te)/total:.1f}%)")

    # Step 4: Oversample (classical only) 
    ros = RandomOverSampler(random_state=SEED)
    Xc_tr_rs, y_tr_rs = ros.fit_resample(
        X_tr["clean_text"].values.reshape(-1, 1), y_tr)
    Xc_tr_rs = pd.Series(Xc_tr_rs.ravel())
    print(f"[ROS] After oversampling train: {len(Xc_tr_rs):,} samples")

    # Step 5: Classical models 
    print("\n Classical Baseline Models ")
    classical_results = run_classical_models(Xc_tr_rs, y_tr_rs,X_te["clean_text"], y_te,class_names)
    all_results = dict(classical_results)

    # Step 6: Transformer models 
    # Each model: raw text, class-weighted loss, same test split as classical
    transformer_configs = {"BERT":       "bert-base-uncased","RoBERTa":    "roberta-base","DistilBERT": "distilbert-base-uncased"}

    for short_name, hf_name in transformer_configs.items():
        print(f"\n── {short_name} ({hf_name}) "
              f"{'─' * max(1, 50 - len(short_name) - len(hf_name))}")
        result = run_transformer(hf_name,X_tr["text"],  y_tr,X_val["text"], y_val,X_te["text"],  y_te,num_labels, class_names)

        if result is not None:
            all_results[short_name] = result   
            print(f"[TRF] {short_name}"
                  f"Acc={result['accuracy']:.4f}"  
                  f"F1={result['f1_weighted']:.4f}")
            plot_confusion_matrix(result["cm"], class_names,f"{short_name} — Confusion Matrix",f"{_safe_filename(short_name)}_confusion_matrix.png")
            plot_training_history(result["history"], short_name)
        else:
            print(f"[WARN] {short_name} did not complete — "f"excluded from comparison table.")

    # Confusion matrices for classical models
    for name, res in classical_results.items():
        plot_confusion_matrix(res["cm"], class_names,f"{name} — Confusion Matrix",f"{_safe_filename(name)}_confusion_matrix.png")

    # Step 7: Final comparison 
    if not all_results:
        print("[ERROR] No models completed:nothing to compare.")
        return

    plot_model_comparison(all_results)
    plot_all_confusion_matrices(all_results, class_names)  
    print_final_table(all_results)         
 
# Save outputs 
    summary = {name: {"accuracy": round(res["accuracy"],4),"f1_weighted": round(res["f1_weighted"],4)}
        for name, res in all_results.items()}
    with open(f"{OUTPUT_DIR}/results_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(f"[OUT] results_summary.json saved.")

    with open(f"{OUTPUT_DIR}/classification_reports.txt", "w") as f:
        for name in MODEL_ORDER:
            if name not in all_results:
                continue
            f.write(all_results[name]["report"])
    print(f"[OUT] classification_reports.txt saved.")

    print(f"  [DONE] All outputs saved to {OUTPUT_DIR}/")
    print(f"  Models compared: {list(all_results.keys())}")
    
if __name__ == "__main__":
    main()

Mental Health Discourse Classification — IEEE Paper Pipeline
6 Models: LR | NB | LinearSVC | DistilBERT | BERT | RoBERTa
[DATA] Loaded 52,680 rows | 7 classes

  TABLE I — Class Distribution
  Category                   Samples       %
-------------------------------------------------------
  Normal                      16,342   31.0
  Depression                  15,404   29.2
  Suicidal                    10,652   20.2
  Anxiety                      3,841    7.3
  Bipolar                      2,777    5.3
  Stress                       2,587    4.9
  Personality disorder         1,077    2.0
-------------------------------------------------------
  Total                       52,680   100.0

[PRE] Cleaning text for classical models …
[PRE] Classes (7): ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']
[SPL] Train=36,875Val=7,903  Test=7,902(70.0% / 15.0% /15.0%)
[ROS] After oversampling train: 80,073 samples

 Classical Baseline Models 
[CLS]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

[TRF] distilbert-base-uncased  |  device=cuda


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  Epoch 1/10  tr_loss=1.2456  val_loss=0.6642  val_acc=0.7474
  Epoch 2/10  tr_loss=0.5733  val_loss=0.5537  val_acc=0.7826
  Epoch 3/10  tr_loss=0.4058  val_loss=0.5185  val_acc=0.8044
  Epoch 4/10  tr_loss=0.2871  val_loss=0.5787  val_acc=0.8151
  Epoch 5/10  tr_loss=0.2107  val_loss=0.6343  val_acc=0.8210
  Epoch 6/10  tr_loss=0.1532  val_loss=0.6947  val_acc=0.8221
  Epoch 7/10  tr_loss=0.1113  val_loss=0.7649  val_acc=0.8192
  Epoch 8/10  tr_loss=0.0832  val_loss=0.8697  val_acc=0.8174
  Epoch 9/10  tr_loss=0.0638  val_loss=0.9280  val_acc=0.8178
  Epoch 10/10  tr_loss=0.0515  val_loss=0.9444  val_acc=0.8188
[TRF] distilbert-base-uncased  TEST → Acc=0.8188  F1=0.8192
[TRF] DistilBERTAcc=0.8188F1=0.8192
[PLT] model_comparison.png saved.
[PLT] all_confusion_matrices.png saved  (6/6 models).

  TABLE II — Model Performance on Test Set
  Model                     Accuracy    F1 (wtd)    Params
--------------------------------------------------------------------
  Naive Bayes          